# Ridge sans clipping

Étape 1 : charger les données sans imputation et vérifier la cible.


In [6]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score


In [7]:
# Les données sont dans le dossier parent du notebook.
data_dir = Path("..")

# La colonne id devient l'index : elle ne sera pas une variable du modèle.
X = pd.read_csv(data_dir / "X_train.csv", index_col="id")
y = pd.read_csv(data_dir / "y_train.csv", index_col="id")["y"]

assert X.index.is_unique and y.index.is_unique, "Identifiants dupliqués"
assert X.index.equals(y.index), "X et y ne sont pas alignés"
assert y.notna().all(), "La cible contient des valeurs manquantes"

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Valeurs manquantes dans X :", X.isna().sum().sum())
print("Nombre de valeurs distinctes de y :", y.nunique())
display(y.describe())
display(X.head())


Dimensions de X : (1212, 832)
Dimensions de y : (1212,)
Valeurs manquantes dans X : 76902
Nombre de valeurs distinctes de y : 56


count    1212.000000
mean       69.889439
std         9.720843
min        42.000000
25%        64.000000
50%        70.000000
75%        77.000000
max        97.000000
Name: y, dtype: float64

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x822,x823,x824,x825,x826,x827,x828,x829,x830,x831
id,,,,,,,,,,,,,,,,,,,,,
0.0,14168.823171,10514.380717,3316.149698,94230.695124,102.386606,92.677127,11108.748199,10866.505510,10837.622093,10.227734,...,NaN,12352.094085,846.014651,105.132144,102.112809,2090.004260,2.691845,1234.374109,1000.784475,9285.751272
1.0,17757.037554,NaN,4101.016273,92959.527633,NaN,99.855168,10013.959449,10826.607494,10076.101597,11.436970,...,NaN,16198.071494,776.084467,106.385590,103.472030,2474.051881,2.287976,NaN,1012.626705,11750.284764
2.0,14226.656663,11029.642499,NaN,124055.600561,100.542483,92.860892,NaN,10492.342868,NaN,10.810076,...,10329.704431,13976.063780,737.040332,103.671234,109.458246,2656.083281,2.843706,888.353607,1048.810385,9553.922728
3.0,8766.012436,7384.202998,2147.308418,100157.719990,104.855061,101.929026,10050.049932,10499.521099,10525.030989,10.092109,...,10008.251395,6212.127347,329.044233,105.084488,104.858546,1097.785204,2.732257,927.752967,1048.357330,NaN
4.0,13801.016418,13269.493652,3408.316953,92048.527786,103.759758,95.789235,9667.353978,10750.783106,10618.800750,12.006773,...,10095.782015,13772.061493,NaN,NaN,100.369834,2693.053231,2.702908,1471.354073,1071.284484,9423.533063


Les NaN sont conservés : les médianes seront apprises dans chaque fold par la pipeline. La cible est numérique, avec des valeurs comprises entre 42 et 97 ; le sens de la cible dans l’énoncé permet de confirmer le choix de la régression.


In [8]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("constant_filter", VarianceThreshold()),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10)) 
])

In [9]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("RMSE par fold :", rmse)
print(f"RMSE moyenne : {rmse.mean():.3f}")
print(f"Écart-type : {rmse.std():.3f}")

RMSE par fold : [14.26123739 13.7596251  15.04874416 15.06372777 14.49307946]
RMSE moyenne : 14.525
Écart-type : 0.494


### Première baseline : Ridge sans clipping

Le jeu d’entraînement contient 1 212 observations et 832 variables
explicatives, après retrait de l’identifiant. Environ 7,6 % des valeurs
explicatives sont manquantes. La cible varie de 42 à 97, avec une moyenne
de 69,89 et un écart-type de 9,72.

Nous évaluons une première approche sans traitement des outliers :
- imputation des valeurs manquantes par la médiane ;
- suppression des colonnes constantes après imputation ;
- standardisation des variables ;
- régression Ridge avec alpha = 10.

Ces étapes sont regroupées dans une pipeline et ajustées uniquement
sur les données d’entraînement de chaque fold, pour éviter les fuites
d’information.

### Résultats de la validation croisée

Avec 5 folds mélangés et random_state = 42 :
- RMSE moyenne : 14,525 ;
- écart-type des RMSE entre folds : 0,494 ;
- RMSE individuelles comprises entre 13,76 et 15,06.

Les erreurs sont relativement proches entre les folds. Cependant, leur
niveau paraît élevé par rapport à la dispersion de la cible (9,72).
Cette comparaison suggère que le modèle pourrait être moins performant
qu’une simple prédiction par la moyenne ; elle doit être vérifiée avec
une évaluation sur les mêmes folds.

### Comparaison avec la prédiction par la moyenne

Nous allons évaluer un DummyRegressor qui calcule la moyenne de y sur
le train de chaque fold, puis prédit cette même valeur pour toutes les
observations de validation.

Cette référence permet de vérifier si Ridge exploite les variables
mieux qu’une règle qui les ignore complètement.

### Prochaines étapes

1. Mesurer la RMSE du DummyRegressor sur les mêmes folds.
2. Comparer cette RMSE à celle de Ridge.
3. Tester plusieurs valeurs de alpha, notamment plus élevées, pour
   vérifier si une régularisation plus forte améliore Ridge.
4. Comparer ensuite d’autres traitements, une modification à la fois :
   RobustScaler, clipping, sélection de variables ou PCA.

À ce stade, nous ne pouvons pas attribuer la mauvaise performance aux
outliers. Le nombre élevé de variables par rapport aux observations et
une régularisation insuffisante sont des pistes à examiner.

Les scores utilisés pour choisir les traitements servent à la mise au
point du modèle ; une évaluation indépendante sera nécessaire pour
estimer la performance finale sans biais de sélection.

In [11]:
from sklearn.dummy import DummyRegressor

dummy_rmse = -cross_val_score(
    DummyRegressor(strategy="mean"),
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

print("RMSE par fold :", dummy_rmse)
print(f"RMSE moyenne Dummy : {dummy_rmse.mean():.3f}")
print(f"Écart-type : {dummy_rmse.std():.3f}")
print(f"RMSE moyenne Ridge : {rmse.mean():.3f}")

RMSE par fold : [ 9.4993167   9.36650628  9.65932935 10.27874636  9.79385752]
RMSE moyenne Dummy : 9.720
Écart-type : 0.315
RMSE moyenne Ridge : 14.525


Comparaison avec R2

In [13]:
ridge_r2 = cross_val_score(
    pipeline, X, y,
    cv=cv,
    scoring="r2"
)

dummy_r2 = cross_val_score(
    DummyRegressor(strategy="mean"), X, y,
    cv=cv,
    scoring="r2"
)

print("R² Ridge par fold :", ridge_r2)
print(f"R² moyen Ridge : {ridge_r2.mean():.3f}")

print("\nR² Dummy par fold :", dummy_r2)
print(f"R² moyen Dummy : {dummy_r2.mean():.3f}")

R² Ridge par fold : [-1.26036113 -1.15851223 -1.45310925 -1.14858934 -1.20549084]
R² moyen Ridge : -1.245

R² Dummy par fold : [-0.0028781  -0.00022238 -0.0106722  -0.00038875 -0.00714378]
R² moyen Dummy : -0.004


Comparaison avec différents alpha 

In [14]:
from sklearn.base import clone

alphas = [0.1, 1, 10, 100, 1000, 10000, 100000]
results = []

for alpha in alphas:
    model = clone(pipeline).set_params(ridge__alpha=alpha)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="r2"
    )

    results.append({
        "alpha": alpha,
        "R2_mean": scores.mean(),
        "R2_std": scores.std()
    })

results = pd.DataFrame(results)
display(results.sort_values("R2_mean", ascending=False))

,alpha,R2_mean,R2_std
4,1000.0,0.293513,0.087058
5,10000.0,0.269406,0.057303
6,100000.0,0.116822,0.022802
3,100.0,-0.094569,0.108249
2,10.0,-1.245213,0.111249
1,1.0,-2.838079,0.113959
0,0.1,-3.449988,0.220041


Ridge avec Feature Selection en plus 

In [16]:
from sklearn.feature_selection import SelectKBest, f_regression

selection_results = []

for k in [25, 50, 100, 200, 400, "all"]:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("constant_filter", VarianceThreshold()),
        ("scaler", StandardScaler()),
        ("selection", SelectKBest(score_func=f_regression, k=k)),
        ("ridge", Ridge(alpha=1000))
    ])

    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring="r2"
    )

    selection_results.append({
        "k": k,
        "R2_mean": scores.mean(),
        "R2_std": scores.std()
    })

selection_results = pd.DataFrame(selection_results)
display(selection_results.sort_values("R2_mean", ascending=False))

,k,R2_mean,R2_std
3,200,0.356274,0.070087
2,100,0.345014,0.067849
1,50,0.317339,0.067966
4,400,0.304454,0.082823
0,25,0.297204,0.066643
5,all,0.293513,0.087058


Pipeline avec clipping 

In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        # Ignore les NaN pour calculer les seuils
        self.lower_bounds_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)

In [18]:
clipping_options = [
    ("Sans clipping", "passthrough"),
    ("Clipping 1–99 %", QuantileClipper(0.01, 0.99)),
    ("Clipping 0,5–99,5 %", QuantileClipper(0.005, 0.995)),
]

clipping_results = []

for name, clipper in clipping_options:
    model = Pipeline([
        ("clipper", clipper),
        ("imputer", SimpleImputer(strategy="median")),
        ("constant_filter", VarianceThreshold()),
        ("scaler", StandardScaler()),
        ("selection", SelectKBest(score_func=f_regression, k=200)),
        ("ridge", Ridge(alpha=1000))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="r2"
    )

    clipping_results.append({
        "traitement": name,
        "R2_mean": scores.mean(),
        "R2_std": scores.std()
    })

clipping_results = pd.DataFrame(clipping_results)
display(clipping_results.sort_values("R2_mean", ascending=False))

,traitement,R2_mean,R2_std
1,Clipping 1–99 %,0.389227,0.069894
2,"Clipping 0,5–99,5 %",0.376730,0.067588
0,Sans clipping,0.356274,0.070087


Test des différentes méthodes d'imputation

In [19]:
imputation_options = [
    ("Médiane", SimpleImputer(strategy="median")),
    ("Moyenne", SimpleImputer(strategy="mean")),
    ("Médiane + indicateurs NaN",
     SimpleImputer(strategy="median", add_indicator=True)),
    ("Moyenne + indicateurs NaN",
     SimpleImputer(strategy="mean", add_indicator=True)),
]

imputation_results = []

for name, imputer in imputation_options:
    model = Pipeline([
        ("clipper", QuantileClipper(0.01, 0.99)),
        ("imputer", imputer),
        ("constant_filter", VarianceThreshold()),
        ("scaler", StandardScaler()),
        ("selection", SelectKBest(score_func=f_regression, k=200)),
        ("ridge", Ridge(alpha=1000))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="r2"
    )

    imputation_results.append({
        "imputation": name,
        "R2_mean": scores.mean(),
        "R2_std": scores.std()
    })

imputation_results = pd.DataFrame(imputation_results)

display(
    imputation_results.sort_values("R2_mean", ascending=False)
)

,imputation,R2_mean,R2_std
0,Médiane,0.389227,0.069894
1,Moyenne,0.389180,0.070002
2,Médiane + indicateurs NaN,0.385091,0.069289
3,Moyenne + indicateurs NaN,0.384607,0.069635


Tuning de alpha et k avec clipping 

In [20]:
from sklearn.model_selection import GridSearchCV

ridge_pipeline = Pipeline([
    ("clipper", QuantileClipper(0.01, 0.99)),
    ("imputer", SimpleImputer(strategy="median")),
    ("constant_filter", VarianceThreshold()),
    ("scaler", StandardScaler()),
    ("selection", SelectKBest(score_func=f_regression)),
    ("ridge", Ridge())
])

param_grid = {
    "selection__k": [50, 100, 200, 400, "all"],
    "ridge__alpha": [10, 100, 300, 1000, 3000, 10000]
}

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True
)

search.fit(X, y)

print("Meilleurs paramètres :", search.best_params_)
print(f"Meilleur R² moyen : {search.best_score_:.4f}")

Meilleurs paramètres : {'ridge__alpha': 300, 'selection__k': 200}
Meilleur R² moyen : 0.3957


In [21]:
grid_results = pd.DataFrame(search.cv_results_)

display(
    grid_results[
        [
            "param_selection__k",
            "param_ridge__alpha",
            "mean_test_score",
            "std_test_score"
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .head(10)
)

,param_selection__k,param_ridge__alpha,mean_test_score,std_test_score
12,200,300,0.395731,0.072252
17,200,1000,0.389227,0.069894
11,100,300,0.388097,0.066868
6,100,100,0.384530,0.069987
7,200,100,0.373056,0.070274
16,100,1000,0.370051,0.063467
1,100,10,0.359431,0.082718
10,50,300,0.355719,0.063731
5,50,100,0.355646,0.064741
22,200,3000,0.354273,0.062415


Validation imbriquée

In [23]:
from sklearn.model_selection import cross_validate
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=123)

inner_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
    refit=True
)

nested_results = cross_validate(
    inner_search,
    X,
    y,
    cv=outer_cv,
    scoring={
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error"
    },
    return_estimator=True,
    n_jobs=1
)

In [24]:
fold_results = pd.DataFrame({
    "fold": range(1, 6),
    "R2": nested_results["test_r2"],
    "RMSE": -nested_results["test_rmse"],
    "MAE": -nested_results["test_mae"],
    "k": [
        estimator.best_params_["selection__k"]
        for estimator in nested_results["estimator"]
    ],
    "alpha": [
        estimator.best_params_["ridge__alpha"]
        for estimator in nested_results["estimator"]
    ]
})

display(fold_results)

print(
    f"R² externe moyen : {fold_results['R2'].mean():.4f}"
    f" ± {fold_results['R2'].std(ddof=0):.4f}"
)
print(f"RMSE externe moyenne : {fold_results['RMSE'].mean():.3f}")
print(f"MAE externe moyenne : {fold_results['MAE'].mean():.3f}")

,fold,R2,RMSE,MAE,k,alpha
0,1,0.344879,7.521874,5.393196,200,300
1,2,0.491911,6.319363,4.670497,200,300
2,3,0.343063,8.634056,6.530457,100,300
3,4,0.353371,8.250598,5.982505,200,300
4,5,0.401575,7.211126,5.469796,200,300


R² externe moyen : 0.3870 ± 0.0567
RMSE externe moyenne : 7.587
MAE externe moyenne : 5.609


Importation de X_test, création d'une submission 

In [26]:
X_test = pd.read_csv(data_dir / "X_test.csv", index_col="id")
submission = pd.read_csv(data_dir / "sample.csv")

assert set(X_test.columns) == set(X.columns)
X_test = X_test.loc[:, X.columns]

predictions = pd.Series(
    search.best_estimator_.predict(X_test),
    index=X_test.index
)

submission["y"] = submission["id"].map(predictions)

assert len(submission) == len(X_test)
assert np.isfinite(submission["y"]).all()

submission.to_csv(data_dir / "submission_ridge.csv", index=False)

display(submission.head())

,id,y
0,0.0,63.943053
1,1.0,71.311949
2,2.0,67.723213
3,3.0,76.220785
4,4.0,74.327691
